# ML-07 — Baseline Action Score and Top-20 Review

## 1. My rule and its reason codes

**Lane 2 — Refresh / Content Opportunity Scoring.**

I will use two observable signals: **staleness** (`days_since_last_update`) and **search visibility** (`impressions_90d`). Staleness is linked to FlyRank's refresh/staleness logic, while volume/visibility is linked to the quick-win logic.

**Rule:** prioritize a page when it is at least 180 days since its last update **and** has at least 500 search impressions in the observed 90-day window. Among those pages, rank by impressions so the reviewer sees the more visible opportunities first.

**Score:** `stale × visible × impressions_90d`.

**Reason code:** `stale_and_visible`. Pages that do not meet both conditions receive score 0 and remain below the actionable queue.

**Action label:** `review_refresh`. This does not mean the page definitely needs a refresh; it means it is worth reviewing earlier.

Both signals are available in the starter snapshot and do not use future-window outcomes or the declining label.

In [19]:
import pandas as pd

# The starter dataset is the baseline's fixed, reproducible snapshot.
df = pd.read_csv('/content/content_refresh_anonymized.csv')

# Signal 1: staleness. Bucket counts are printed with n.
df['stale_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 30, 90, 180, float('inf')],
    labels=['0-30', '31-90', '91-180', '181+']
)
stale_table = (
    df.groupby('stale_bucket', observed=False)
      .size()
      .reset_index(name='n')
)
print('Signal 1 — staleness bucket table')
print(stale_table.to_string(index=False))
print('Verdict: CONFIRMED — the signal has multiple observed buckets, including a distinct stale tail.')

# Signal 2: search visibility / volume.
df['visibility_bucket'] = pd.cut(
    df['impressions_90d'],
    bins=[-1, 99, 499, 2999, 29999, float('inf')],
    labels=['0-99', '100-499', '500-2,999', '3,000-29,999', '30,000+']
)
visibility_table = (
    df.groupby('visibility_bucket', observed=False)
      .size()
      .reset_index(name='n')
)
print('\nSignal 2 — search visibility bucket table')
print(visibility_table.to_string(index=False))
print('Verdict: CONFIRMED — the signal is present with meaningful variation in the dataset and is suitable for testing in this baseline. This does not yet show that it is predictive of a successful refresh.')

Signal 1 — staleness bucket table
stale_bucket     n
        0-30 20480
       31-90   175
      91-180  9171
        181+   174
Verdict: CONFIRMED — the signal has multiple observed buckets, including a distinct stale tail.

Signal 2 — search visibility bucket table
visibility_bucket    n
             0-99 7994
          100-499 5280
        500-2,999 8443
     3,000-29,999 7205
          30,000+ 1078
Verdict: CONFIRMED — the signal is present with meaningful variation in the dataset and is suitable for testing in this baseline. This does not yet show that it is predictive of a successful refresh.


## 2. Build the ranked queue (writes the CSV)

The baseline is intentionally transparent and frozen: no fitted weights, no label-derived inputs, and no future-window measurements. The score is a simple multiplication of the two conditions and observed impressions. The output contains the page ID, score, reason code, and action label so a human can understand why an item was ranked.

In [20]:
# Encode ONE transparent rule.

df['stale'] = (df['days_since_last_update'] >= 180).astype(int)
df['visible'] = (df['impressions_90d'] >= 500).astype(int)

df['baseline_score'] = (
    df['stale'] * df['visible'] * df['impressions_90d']
)

df['reason_code'] = 'not_selected'
df.loc[(df['stale'] == 1) & (df['visible'] == 1), 'reason_code'] = 'stale_and_visible'

df['action'] = 'monitor'
df.loc[df['baseline_score'] > 0, 'action'] = 'review_refresh'

queue = (
    df[['content_id', 'client_id', 'baseline_score', 'reason_code', 'action',
        'days_since_last_update', 'impressions_90d']]
    .sort_values(['baseline_score', 'impressions_90d'], ascending=False)
    .reset_index(drop=True)
)
queue['rank'] = queue.index + 1

# Keep the required ranked output columns first.
queue = queue[['rank', 'content_id', 'client_id', 'baseline_score',
               'reason_code', 'action', 'days_since_last_update', 'impressions_90d']]
from pathlib import Path

output_path = Path("/content/work/outputs/baseline_action_score.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

queue.to_csv(output_path, index=False)

print("Rows in ranked queue:", len(queue))
print("Actionable pages:", (queue["baseline_score"] > 0).sum())
print("Written:", output_path)
print("\nTop 10:")
display(queue.head(10))

Rows in ranked queue: 30000
Actionable pages: 17
Written: /content/work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_and_visible,review_refresh,194,61678
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_and_visible,review_refresh,194,59472
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_and_visible,review_refresh,194,25715
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_and_visible,review_refresh,193,13299
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_and_visible,review_refresh,194,7812
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_and_visible,review_refresh,193,7558
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_and_visible,review_refresh,194,4590
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_and_visible,review_refresh,194,4556
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_and_visible,review_refresh,194,4429
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_and_visible,review_refresh,193,1697


## 3. Top-20 review

The top 20 below are not automatically accepted as correct recommendations. Each needs a human review. The code generates a review table with the action, reason code, confidence note, and a specific condition that could make the recommendation wrong.

In [23]:
top20 = queue.head(20).copy()

def confidence_note(action):
    if action == "review_refresh":
        return "Moderate: meets both baseline thresholds; review evidence before acting."
    return "Low: did not meet both baseline thresholds; included only because the top 20 extends beyond the actionable queue."

def wrong_reason(action):
    if action == "review_refresh":
        return "The page may still be performing well, the traffic may be low-quality, or the apparent staleness may be acceptable for this topic."
    return "The fixed thresholds may have excluded a page that deserves review for another reason."

top20["confidence_note"] = top20["action"].apply(confidence_note)
top20["what_would_make_it_wrong"] = top20["action"].apply(wrong_reason)

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

print(
    "\nTop-20 review requirement: each row has action, reason, "
    "confidence note, and what would make it wrong."
)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
1,2,content_7368877ea310,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
2,3,content_1bfaa38ff26c,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
3,4,content_0a91db491d14,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
4,5,content_5feee3994adb,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
5,6,content_c2d929d83eaa,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
6,7,content_b16bd7307b39,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
7,8,content_fe16a55cd13d,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
8,9,content_ecb6215e79fd,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."
9,10,content_928af3e22c80,review_refresh,stale_and_visible,Moderate: meets both baseline thresholds; revi...,"The page may still be performing well, the tra..."



Top-20 review requirement: each row has action, reason, confidence note, and what would make it wrong.


## 4. Weak picks + leakage check

A useful baseline review should identify at least one plausible weak pick rather than assuming the rule is perfect. I will inspect the lowest-ranked selected pages and record why they might be weak.

The rule uses only `days_since_last_update` and `impressions_90d`. It does **not** use `trend_direction`, `trend_pct`, `is_declining_label`, product flags, or any future-window outcome.


In [22]:
# Inspect the weakest selected pages.
weak_picks = queue[queue['baseline_score'] > 0].tail(5)
print('Weakest selected pages:')
display(weak_picks)

# Explicit leakage audit.
forbidden = ['trend_direction', 'trend_pct', 'is_declining_label']
used_rule_inputs = {'days_since_last_update', 'impressions_90d'}
leaked = used_rule_inputs.intersection(forbidden)
print('\nLeakage inputs used by rule:', leaked)
assert not leaked
print('Leakage check: PASS — no label-derived or future-window input is used.')

Weakest selected pages:


,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,impressions_90d
12,13,content_7f116ae1f6f5,client_9400f1b21c,954,stale_and_visible,review_refresh,301,954
13,14,content_77d4d5930e5e,client_7f2253d7e2,828,stale_and_visible,review_refresh,194,828
14,15,content_72496874f806,client_4ec9599fc2,821,stale_and_visible,review_refresh,301,821
15,16,content_6226ee6adc91,client_d029fa3a95,545,stale_and_visible,review_refresh,183,545
16,17,content_074ba6ead17b,client_d029fa3a95,533,stale_and_visible,review_refresh,183,533



Leakage inputs used by rule: set()
Leakage check: PASS — no label-derived or future-window input is used.


## 5. Self-check

- [x] Two signal checks with bucket tables and n are included; both have explicit verdicts.
- [x] At least one signal is linked to a real FlyRank flag/logic: staleness is linked to refresh logic and visibility is linked to quick-win logic.
- [x] Exactly one transparent rule is encoded with a score, one reason code, and an action label.
- [x] The ranked queue is written to `work/outputs/baseline_action_score.csv` when the notebook runs.
- [x] Top-20 review includes action, reason, confidence note, and what would make each pick wrong.
- [x] Weak picks are inspected.
- [x] No future-window or label-derived inputs are used.
- [x] Run all cells in Colab and save the executed notebook before committing.
